# 2.12-张量与自动求导实验

张量保存数据，自动求导计算损失对参数的梯度，优化器利用梯度更新参数。

In [ ]:
import torch
from torch import nn

torch.manual_seed(42)
print('PyTorch:', torch.__version__)

## 2.12.1-张量的形状、类型与设备

| 属性 | 回答的问题 | MNIST 中的例子 |
| --- | --- | --- |
| `shape` | 有几条轴，每条轴多长？ | `[N, 1, 28, 28]`：批量、灰度通道、高、宽 |
| `dtype` | 元素是什么数值类型？ | 图像 `float32`，类别标签 `int64` |
| `device` | 数据位于哪种设备？ | `cpu`、`cuda:0` 或 `mps` |

张量可以理解为多维数组。维度数量与每个维度的长度不同：`[2, 1, 3, 3]` 是 **4 维张量**，并不是“2 维”。
下面用 2 张小图模拟一个批次。除以 17 只是让本例像素位于 `[0, 1]`，并不是 MNIST 的正式预处理。

In [ ]:
images = torch.arange(18, dtype=torch.float32).reshape(2, 1, 3, 3) / 17
labels = torch.tensor([0, 1], dtype=torch.int64)
for name, value in [('images', images), ('labels', labels)]:
    print(name, 'shape=', tuple(value.shape), 'dtype=', value.dtype, 'device=', value.device)

assert images.shape == (2, 1, 3, 3)
assert labels.shape == (2,)
assert images.dtype == torch.float32 and labels.dtype == torch.int64

## 2.12.2-展平保留批量轴，线性层改变特征数

`nn.Flatten(start_dim=1)` 将每张图的通道、高、宽合并，保留第 0 维的样本数：
`[2, 1, 3, 3] → [2, 9]`。线性层 `Linear(9, 4)` 再将 9 个输入特征映射为 4 个输出特征。

`Linear` 对每条样本计算 $y = xW^T+b$。权重形状为 `[输出特征数, 输入特征数]`。
`layer(flat)` 会进入模块的标准调用流程；不用手动调用 `layer.forward(flat)`。
模型参数和输入应位于兼容设备、具有兼容类型；单独对输入调用 `.to(...)` 不会自动移动模型。

批量输入可以写成 $X\in\mathbb R^{N\times d}$，权重为 $W\in\mathbb R^{m\times d}$，偏置为 $b\in\mathbb R^m$。输出 $Y=XW^T+b$ 的形状为 $[N,m]$：每条样本单独经过同一个线性变换，偏置沿批量轴广播。

In [ ]:
device = torch.device('cpu')
flatten = nn.Flatten(start_dim=1)
layer = nn.Linear(9, 4).to(device)
flat = flatten(images.to(device))
features = layer(flat)

print('展平:', tuple(flat.shape))
print('权重:', tuple(layer.weight.shape), '偏置:', tuple(layer.bias.shape))
print('输出:', tuple(features.shape))
assert flat.shape == (2, 9) and features.shape == (2, 4)

## 2.12.3-`backward()` 求导，梯度会累加

令 $f(w)=(w-3)^2$，则 $f'(w)=2(w-3)$；当 $w=1$，梯度应为 $-4$。
只有需要求导的浮点或复数张量才使用 `requires_grad=True`，分类标签无需梯度。

每次重新计算损失都会建立新的计算图。下面两次反向传播之间没有清空 `w.grad`，因此梯度相加得到 $-8$。
这演示的是**梯度缓冲区的累加**；若对同一个已释放的计算图直接再次 `backward()`，通常会报错。

对标量损失 $L$，参数 $w$ 的梯度 $\partial L/\partial w$ 描述损失对参数微小变化的敏感程度。反向传播沿计算图应用链式法则，把输出端的梯度逐层传回参数；求得梯度本身不会改变参数值。

In [ ]:
w = torch.tensor(1.0, requires_grad=True)
loss = (w - 3).square()
loss.backward()
first_gradient = w.grad.detach().clone()

new_loss = (w - 3).square()  # 新的前向计算，新的计算图
new_loss.backward()
accumulated_gradient = w.grad.detach().clone()

print('第一次梯度:', first_gradient.item())
print('累计梯度:', accumulated_gradient.item())
torch.testing.assert_close(first_gradient, torch.tensor(-4.0))
torch.testing.assert_close(accumulated_gradient, torch.tensor(-8.0))

## 2.12.4-清梯度、求梯度、更新参数是不同的动作

用单个参数和 SGD 看清一次更新：`zero_grad → forward → loss → backward → step`。
学习率为 $0.1$ 时，本例第一次更新为 $1-0.1\times(-4)=1.4$。

`optimizer.zero_grad(set_to_none=True)` 将已有梯度设为 `None`，而不是把模型权重归零。
`loss.backward()` 写入梯度；`optimizer.step()` 使用梯度修改权重。这里用 SGD 观察梯度下降的更新公式；后面的 MNIST 实验使用 Adam。

不含动量和权重衰减时，SGD 的单步更新为 $w_{t+1}=w_t-\eta\nabla_w L$，其中 $\eta$ 是学习率。减去梯度使参数沿损失局部下降的方向移动；步长过大仍可能越过较低损失的位置。

In [ ]:
parameter = nn.Parameter(torch.tensor(1.0))
optimizer = torch.optim.SGD([parameter], lr=0.1)

optimizer.zero_grad(set_to_none=True)
assert parameter.grad is None
loss = (parameter - 3).square()
loss.backward()
gradient_before_step = parameter.grad.detach().clone()
optimizer.step()

print('使用的梯度:', gradient_before_step.item())
print('更新后的参数:', parameter.item())
torch.testing.assert_close(parameter.detach(), torch.tensor(1.4))
optimizer.zero_grad(set_to_none=True)
assert parameter.grad is None

## 2.12.5-`no_grad()` 不等于清梯度，也不等于 `eval()`

`torch.no_grad()` 让其中的运算不建立反向传播记录，适合不需要求导的预测。
它不会删除已有的 `.grad`。`model.eval()` 则切换模块的训练/评估行为，例如 Dropout、BatchNorm；
它本身不关闭自动求导。实际评估通常需要同时使用评估模式和禁用梯度的上下文。

下面保留刚算出的梯度，进入 `no_grad()` 后验证：结果不需要梯度，但原来的梯度仍在。
MNIST 中会使用更严格的 `torch.inference_mode()` 完成纯推理。

| 操作 | 改变什么 | 不会自动完成什么 |
| --- | --- | --- |
| `model.eval()` | Dropout、BatchNorm 等层的行为 | 关闭梯度记录 |
| `torch.no_grad()` | 上下文内运算的梯度记录 | 切换模型模式、清空已有梯度 |
| `optimizer.zero_grad()` | 优化器所管理参数的梯度 | 更新参数 |

线性层在训练和评估模式下计算相同，所以本单元的 `layer.eval()` 用于检查模式切换与梯度记录是否相互独立。

In [ ]:
tracked = torch.tensor(2.0, requires_grad=True)
tracked.square().backward()
saved_grad = tracked.grad.clone()
with torch.no_grad():
    prediction = tracked * 3

print('预测结果是否求导:', prediction.requires_grad)
print('原参数已有梯度:', tracked.grad.item())
assert not prediction.requires_grad
torch.testing.assert_close(tracked.grad, saved_grad)

layer.eval()
still_tracked = layer(flat)
with torch.no_grad():
    untracked = layer(flat)
assert still_tracked.requires_grad and not untracked.requires_grad

## 2.12.6-多分类：logits、概率、类别与交叉熵

本例有 2 条样本、3 个类别。模型原始输出 `logits` 形状为 `[2, 3]`，不要求介于 0 和 1。
类别标签形状为 `[2]`、类型为 `int64`，取值为 `0、1、2`。

对于这里使用的整数类别标签，`CrossEntropyLoss` 直接接收 logits，内部等价于 `log_softmax` 后计算负对数似然。
因此训练前不要再手动 softmax。`softmax(dim=1)` 可以在展示预测时计算各类别概率；
`argmax(dim=1)` 选出每条样本得分最大的类别。最大概率并不是“预测正确”的保证，也不表示已校准置信度。


对于一条样本，设第 $k$ 类的 logit 为 $z_k$，真实类别为 $y$，则：

$$
p_k=\frac{e^{z_k}}{\sum_j e^{z_j}},\qquad L=-\log p_y.
$$

交叉熵关注模型给真实类别分配的概率。真实类别的概率越小，损失越大。把 logits 直接传给 `CrossEntropyLoss`，可以由框架在数值稳定的形式下合并计算这些步骤。

In [ ]:
logits = torch.tensor([[2.0, 1.0, -1.0], [-1.0, 0.5, 3.0]], requires_grad=True)
targets = torch.tensor([0, 2], dtype=torch.long)
loss_fn = nn.CrossEntropyLoss()
loss = loss_fn(logits, targets)

manual_loss = -logits.log_softmax(dim=1)[torch.arange(2), targets].mean()
probabilities = logits.detach().softmax(dim=1)
predicted_classes = logits.detach().argmax(dim=1)
torch.testing.assert_close(loss, manual_loss)
torch.testing.assert_close(probabilities.sum(dim=1), torch.ones(2))
assert predicted_classes.tolist() == [0, 2]

loss.backward()
print('概率:', probabilities)
print('预测类别:', predicted_classes.tolist())
print('损失:', loss.item(), 'logits 梯度形状:', tuple(logits.grad.shape))